In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vishnutavanam2709/merfish-staq/merfish.h5ad


In [3]:
!pip install scanpy squidpy anndata torch-geometric -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 28.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 85.0 MB/s eta 0:00:00:00:0100:01


In [16]:
 # stage 1 - preprocessing
# load raw data, remove blank probes, save raw counts,
# log transform, run PCA

import os
import scanpy as sc
import squidpy as sq
import scipy.sparse as sp
import warnings
warnings.filterwarnings("ignore")

# RAW_DATA  = "/kaggle/input/datasets/vishnutavanam2709/merfish-staq/merfish.h5ad"
SAVE_DIR  = "/kaggle/working"
adata=sq.datasets.merfish()
adata=adata[adata.obs.Bregma==-9]
print("--- Stage 1: Preprocessing ---")

# adata = sc.read_h5ad(RAW_DATA)
print(f"loaded: {adata.n_obs} cells x {adata.n_vars} genes")

# remove blank probes (noise, not real genes)
blank_mask = adata.var_names.str.startswith("Blank")
adata = adata[:, ~blank_mask].copy()
print(f"after removing blanks: {adata.n_vars} genes")

# save raw counts before any transform (needed for decoder later)
if sp.issparse(adata.X):
    adata.X = adata.X.toarray().astype("float32")
adata.layers["counts"] = adata.X.copy()

# log transform
sc.pp.log1p(adata, base=2)
adata.layers["x_hat_hvg"] = adata.X.copy()
print("log2(1+x) applied, saved as x_hat")

# PCA
sc.tl.pca(adata, n_comps=30)
print(f"PCA done: {adata.obsm['X_pca'].shape}")

adata.write_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
print("[SUCCESS] saved stage1.h5ad")

--- Stage 1: Preprocessing (No HVG) ---
loaded: 73655 cells x 161 genes
after removing blanks: 156 genes
log2(1 + x) applied
PCA done on all genes: (73655, 30)
x_hat (encoder input): (73655, 156)
[SUCCESS] saved stage1.h5ad


In [17]:
# stage 2 - build two knn graphs
# W_T: cells similar in gene expression (PCA space)
# W_S: cells close in physical space

import os
import numpy as np
import anndata
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"

print("--- Stage 2: kNN Graphs ---")

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
Z = adata.obsm["X_pca"].astype("float32")
S = adata.obsm["spatial"].astype("float32")
n = Z.shape[0]


def build_knn_graph(features, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, n_jobs=-1)
    nbrs.fit(features)
    # print(nbrs)
    distances, indices = nbrs.kneighbors(features)
    # print(distances)
    # print(indices)
    # adaptive bandwidth = distance to k/2-th neighbour
    sigma = distances[:, k//2]
    sigma[sigma == 0] = 1e-8

    row = np.repeat(np.arange(features.shape[0]), k)
    col = indices[:, 1:].flatten()
    dist = distances[:, 1:].flatten()

    weights = np.exp(-(dist**2) / (sigma[row] * sigma[col]))
    W = sp.csr_matrix((weights, (row, col)), shape=(features.shape[0],)*2)
    W = (W + W.T) * 0.5
    return W,distances,indices

print("building W_T (transcriptomic, k=15)...")
W_T,dis_T,ind_T = build_knn_graph(Z, k=15)

print(f"  W_T edges: {W_T.nnz:,}")

print("building W_S (spatial, k=15)...")
W_S,dis_S,ind_S= build_knn_graph(S, k=15)
print(f"  W_S edges: {W_S.nnz:,}")
print(dis_T.shape)
# print(dis_T.head())
print(ind_T[0])

sp.save_npz(os.path.join(SAVE_DIR, "stage2_WT.npz"), W_T)
sp.save_npz(os.path.join(SAVE_DIR, "stage2_WS.npz"), W_S)
print("[SUCCESS] saved stage2_WT.npz, stage2_WS.npz")

--- Stage 2: kNN Graphs ---
building W_T (transcriptomic, k=15)...
  W_T edges: 1,763,806
building W_S (spatial, k=15)...
  W_S edges: 1,221,482
(73655, 16)
[    0 66509 60612 64183 39028 59613 32866   527 67918 68767 61981 66599
 53603 66379 20073 67620]
[SUCCESS] saved stage2_WT.npz, stage2_WS.npz


In [18]:
# stage 3 - niche profile
# for each cell, average the PCA embeddings of its spatial neighbours
# N = D_S^-1 * W_S * Z

import os
import numpy as np
import anndata
import scipy.sparse as sp
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"

print("--- Stage 3: Niche Profile ---")

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
W_S   = sp.load_npz(os.path.join(SAVE_DIR, "stage2_WS.npz"))
Z     = adata.obsm["X_pca"].astype("float32")

deg = np.array(W_S.sum(axis=1)).ravel()
deg[deg == 0] = 1.0

D_inv  = sp.diags(1.0 / deg)
W_norm = D_inv.dot(W_S)
N      = W_norm.dot(Z).astype("float32")

print(f"niche profile N: {N.shape}")

np.save(os.path.join(SAVE_DIR, "stage3_niche.npy"), N)
print("[SUCCESS] saved stage3_niche.npy")

--- Stage 3: Niche Profile ---
niche profile N: (73655, 30)
[SUCCESS] saved stage3_niche.npy


In [19]:
# stage 4 - assemble pytorch tensors
# x_input = [x_hat_hvg || niche]  -> encoder input  (HVG log-counts + niche)
# y_target = raw counts            -> NB decoder target

import os
import numpy as np
import anndata
import scipy.sparse as sp
import torch
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"

print("--- Stage 4: Tensor Assembly ---")

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
W_T   = sp.load_npz(os.path.join(SAVE_DIR, "stage2_WT.npz"))
W_S   = sp.load_npz(os.path.join(SAVE_DIR, "stage2_WS.npz"))
N     = np.load(os.path.join(SAVE_DIR, "stage3_niche.npy"))

# use HVG-subset log-counts as encoder input (not all genes)
x_hat = adata.obsm["x_hat_hvg"].astype("float32")   # (n, n_hvg)

raw = adata.layers["counts"]
if sp.issparse(raw):
    raw = raw.toarray()
raw = raw.astype("float32")

# encoder input: [x_hat_hvg || niche_profile]
x_input = np.concatenate([x_hat, N], axis=1).astype("float32")
print(f"x_input: {x_input.shape}  (n_hvg={x_hat.shape[1]} + d_pca={N.shape[1]})")

def to_edge_index(W):
    W = W.tocoo()
    ei = torch.tensor(np.vstack([W.row, W.col]), dtype=torch.long)
    ew = torch.tensor(W.data, dtype=torch.float32)
    return ei, ew

ei_S, ew_S = to_edge_index(W_S)
ei_T, ew_T = to_edge_index(W_T)

tensors = {
    "x_input":       torch.tensor(x_input, dtype=torch.float32),
    "y_target":      torch.tensor(raw,     dtype=torch.float32),
    "edge_index_S":  ei_S,
    "edge_weight_S": ew_S,
    "edge_index_T":  ei_T,
    "edge_weight_T": ew_T,
}

for k, v in tensors.items():
    print(f"  {k}: {tuple(v.shape)}")

torch.save(tensors, os.path.join(SAVE_DIR, "stage4_tensors.pt"))
print("[SUCCESS] saved stage4_tensors.pt")


--- Stage 4: Tensor Assembly ---
x_input: (73655, 186)  (n_hvg=156 + d_pca=30)
  x_input: (73655, 186)
  y_target: (73655, 156)
  edge_index_S: (2, 1221482)
  edge_weight_S: (1221482,)
  edge_index_T: (2, 1763806)
  edge_weight_T: (1763806,)
[SUCCESS] saved stage4_tensors.pt


In [20]:
# stage 5 - codebook init using stratified fps
# pick M well-spread anchor cells, used as starting codebook

import os
import numpy as np
import anndata
import torch
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"

print("--- Stage 5: Codebook Init (StratifiedFPS) ---")

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
Z = adata.obsm["X_pca"].astype("float32")
S = adata.obsm["spatial"].astype("float32")
n = Z.shape[0]

gamma_M = 75
M = int(np.ceil(n / gamma_M))
beta = 0.5
print(f"n={n}, M={M}")

# estimate median distances from a sample (full pairwise is too slow)
rng = np.random.default_rng(42)
sample = rng.choice(n, 2000, replace=False)
Z_s, S_s = Z[sample], S[sample]

med_T = np.median(np.linalg.norm(Z_s[:, None] - Z_s[None, :], axis=-1))
med_S = np.median(np.linalg.norm(S_s[:, None] - S_s[None, :], axis=-1))
print(f"med_T={med_T:.3f}  med_S={med_S:.3f}")

# seed = cell farthest from mean
seed = int(np.argmax(np.linalg.norm(Z - Z.mean(0), axis=1)))
selected = [seed]
min_d_H = np.linalg.norm(Z - Z[seed], axis=1)
min_d_S = np.linalg.norm(S - S[seed], axis=1)
print(min_d_H.shape)
print(f"running FPS for {M} anchors...")
for step in range(2, M + 1):
    rho = beta * (min_d_H / med_T) + (1 - beta) * (min_d_S / med_S)
    rho[selected] = -np.inf
    # print(rho)
    i_star = int(np.argmax(rho))
    selected.append(i_star)
    min_d_H = np.minimum(min_d_H, np.linalg.norm(Z - Z[i_star], axis=1))
    min_d_S = np.minimum(min_d_S, np.linalg.norm(S - S[i_star], axis=1))

anchor_idx = np.array(selected)
E0 = Z[anchor_idx].astype("float32")
print(f"E0 shape: {E0.shape}")
# print(selected)
torch.save(torch.tensor(E0), os.path.join(SAVE_DIR, "stage5_E0.pt"))
np.save(os.path.join(SAVE_DIR, "stage5_anchors.npy"), anchor_idx)
print("[SUCCESS] saved stage5_E0.pt, stage5_anchors.npy")

--- Stage 5: Codebook Init (StratifiedFPS) ---
n=73655, M=983
med_T=34.678  med_S=0.492
(73655,)
running FPS for 983 anchors...
E0 shape: (983, 30)
[SUCCESS] saved stage5_E0.pt, stage5_anchors.npy


In [ ]:
# # stage 6 - train the vq-spatial autoencoder
# # encoder (GAT) -> codebook (VQ) -> decoder (NB)
# # 5 losses, trained jointly
# import os, math
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torch.optim as optim
# from torch_geometric.nn import GATConv
# from sklearn.cluster import KMeans
# import warnings
# warnings.filterwarnings("ignore")

# D, D_H, N_HEADS = 32, 64, 4
# EPOCHS, PATIENCE, B_SEEDS = 500, 50, 256
# LR = 1e-3
# A_CB, A_CM, A_SP, A_UE = 1.0, 0.25, 1.0, 0.1
# TAU0, T_ANN, N_MIN = 1.0, EPOCHS // 3, 5
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# SAVE_DIR = "/kaggle/working"
# print("--- Stage 6: Training ---")
# print(f"device: {device}")






# # encoder: 2-layer GAT on W_S
# class GATEncoder(nn.Module):
#     def __init__(self, in_dim, d_h=64, d=32, n_heads=4):
#         super().__init__()
#         self.gat1 = GATConv(in_dim, d_h, heads=n_heads, concat=True)
#         self.gat2 = GATConv(n_heads*d_h, d, heads=n_heads, concat=False)

#     def forward(self, x, ei, ew):
#         h = F.elu(self.gat1(x, ei, ew))
#         z = F.elu(self.gat2(h, ei, ew))
#         return z


# # codebook with EMA update
# class VectorQuantizerEMA(nn.Module):
#     def __init__(self, M, d, gamma=0.99, eps=1e-5):
#         super().__init__()
#         self.M, self.d, self.gamma, self.eps = M, d, gamma, eps
#         self.register_buffer("codebook", torch.empty(M, d))
#         self.register_buffer("N", torch.zeros(M))
#         self.register_buffer("Sigma", torch.zeros(M, d))

#     def initialize(self, E0):
#         # FIX 2: ensure E0 is on the same device as the buffers
#         E0 = E0.to(self.codebook.device)
#         self.codebook.copy_(E0)
#         self.N.fill_(1.0)
#         self.Sigma.copy_(E0)

#     def forward(self, z, tau):
#         dists = (z.pow(2).sum(1, keepdim=True) + self.codebook.pow(2).sum(1)
#                  - 2 * z @ self.codebook.t())
#         q_idx = dists.argmin(1)
#         z_q = self.codebook[q_idx]
#         z_tilde = z + (z_q - z).detach()
#         p = F.softmax(-dists / tau, dim=1)
#         loss_cb = F.mse_loss(z.detach(), z_q)
#         loss_cm = F.mse_loss(z, z_q.detach())
#         if self.training:
#             one_hot = torch.zeros(z.size(0), self.M, device=z.device)
#             one_hot.scatter_(1, q_idx.unsqueeze(1), 1)
#             self.N.mul_(self.gamma).add_(one_hot.sum(0), alpha=1-self.gamma)
#             self.Sigma.mul_(self.gamma).add_(one_hot.t() @ z.detach(), alpha=1-self.gamma)
#             self.codebook.data.copy_(self.Sigma / self.N.clamp(min=self.eps).unsqueeze(1))
#         return z_tilde, q_idx, p, loss_cb, loss_cm


# # decoder: MLP -> NB params
# class NBDecoder(nn.Module):
#     def __init__(self, d=32, d_h=64, n_genes=156):
#         super().__init__()
#         self.fc1 = nn.Linear(d, d_h)
#         self.fc2 = nn.Linear(d_h, n_genes)
#         self.theta_star = nn.Parameter(torch.zeros(n_genes))

#     def forward(self, z_tilde, lib_sizes):
#         u = F.elu(self.fc1(z_tilde))
#         rho = F.softmax(self.fc2(u), dim=1)
#         mu = lib_sizes.unsqueeze(1) * rho
#         theta = F.softplus(self.theta_star)
#         return mu, theta


# class STAQModel(nn.Module):
#     def __init__(self, in_dim, M, d=32, d_h=64, n_heads=4, n_genes=156):
#         super().__init__()
#         self.encoder = GATEncoder(in_dim, d_h, d, n_heads)
#         self.quantizer = VectorQuantizerEMA(M, d)
#         self.decoder = NBDecoder(d, d_h, n_genes)

#     def forward(self, x, ei, ew, lib_sizes, tau):
#         z = self.encoder(x, ei, ew)
#         z_tilde, q, p, l_cb, l_cm = self.quantizer(z, tau)
#         mu, theta = self.decoder(z_tilde, lib_sizes)
#         return z, q, p, mu, theta, l_cb, l_cm


# # losses
# def nb_loss(y, mu, theta, eps=1e-8):
#     t1 = torch.lgamma(y+theta+eps) - torch.lgamma(theta+eps) - torch.lgamma(y+1.0)
#     t2 = theta * torch.log(theta / (theta+mu+eps))
#     t3 = y * torch.log(mu / (theta+mu+eps))
#     return -(t1+t2+t3).mean()


# def nb_loss_per_cell(y, mu, theta, eps=1e-8):
#     t1 = torch.lgamma(y+theta+eps) - torch.lgamma(theta+eps) - torch.lgamma(y+1.0)
#     t2 = theta * torch.log(theta / (theta+mu+eps))
#     t3 = y * torch.log(mu / (theta+mu+eps))
#     return -(t1+t2+t3).mean(dim=1)


# def spatial_loss(p, ei, ew, eps=1e-8):
#     p_i, p_j = p[ei[0]], p[ei[1]]
#     m = 0.5 * (p_i + p_j)
#     kl_im = (p_i * (torch.log(p_i+eps) - torch.log(m+eps))).sum(-1)
#     kl_jm = (p_j * (torch.log(p_j+eps) - torch.log(m+eps))).sum(-1)
#     jsd = 0.5*kl_im + 0.5*kl_jm
#     return (ew * jsd).sum() / ei.shape[1]


# def usage_loss(p, eps=1e-8):
#     p_bar = p.mean(0)
#     return (p_bar * torch.log(p_bar+eps)).sum()










In [ ]:
# stage 6 - vq-spatial autoencoder (STAQ)
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GATConv
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore")

D, D_H, H    = 32, 64, 4
EPOCHS       = 300
B            = 256
LR           = 1e-3
A_CB         = 1.0
A_CM         = 0.25
A_SP         = 1.0
A_UE         = 0.1
SP_WARMUP    = 20
TAU0         = 1.0
GAMMA, EPS   = 0.99, 1e-5
N_MIN        = 5
T_ANN        = EPOCHS // 3
LOSS_TOL     = 1e-4
ASSIGN_TOL   = 0.01
WINDOW       = 10
GRAD_CLIP    = 1.0

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "/kaggle/working"
print(f"device: {device}")

class GATEncoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.gat1 = GATConv(in_dim, D_H, heads=H, concat=True)
        self.gat2 = GATConv(H*D_H,  D,   heads=H, concat=False)
        self.bn1  = nn.BatchNorm1d(H * D_H)
        self.bn2  = nn.BatchNorm1d(D)
    def forward(self, x, ei, ew):
        h = F.elu(self.bn1(self.gat1(x, ei, ew)))
        z = F.elu(self.bn2(self.gat2(h, ei, ew)))
        return z

class VectorQuantizerEMA(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.M = M
        self.register_buffer("codebook", torch.empty(M, D))
        self.register_buffer("N",        torch.zeros(M))
        self.register_buffer("Sigma",    torch.zeros(M, D))
    def initialize(self, E0):
        E0 = E0.to(self.codebook.device)
        self.codebook.copy_(E0)
        self.N.fill_(1.0)
        self.Sigma.copy_(E0)
    def forward(self, z, tau):
        dists = (z.pow(2).sum(1, keepdim=True)
                 + self.codebook.pow(2).sum(1)
                 - 2 * z @ self.codebook.t()).clamp(min=0)
        q     = dists.argmin(1)
        z_q   = self.codebook[q]
        z_hat = z + (z_q - z).detach()
        p     = F.softmax(-dists / tau, dim=1)
        l_cb  = F.mse_loss(z.detach(), z_q)
        l_cm  = F.mse_loss(z, z_q.detach())
        if self.training:
            oh = torch.zeros(z.size(0), self.M, device=z.device)
            oh.scatter_(1, q.unsqueeze(1), 1)
            self.N.mul_(GAMMA).add_(oh.sum(0),               alpha=1-GAMMA)
            self.Sigma.mul_(GAMMA).add_(oh.t() @ z.detach(), alpha=1-GAMMA)
            self.codebook.data.copy_(self.Sigma / self.N.clamp(min=EPS).unsqueeze(1))
        return z_hat, q, p, l_cb, l_cm

class NBDecoder(nn.Module):
    def __init__(self, n_genes):
        super().__init__()
        self.fc1        = nn.Linear(D, D_H)
        self.fc2        = nn.Linear(D_H, n_genes)
        self.theta_star = nn.Parameter(torch.full((n_genes,), 2.0))
    def forward(self, z_hat, lib):
        u     = F.elu(self.fc1(z_hat))
        rho   = F.softmax(self.fc2(u), dim=1)
        mu    = lib.unsqueeze(1) * rho
        theta = F.softplus(self.theta_star)
        return mu, theta

class STAQ(nn.Module):
    def __init__(self, in_dim, M, n_genes):
        super().__init__()
        self.enc = GATEncoder(in_dim)
        self.vq  = VectorQuantizerEMA(M)
        self.dec = NBDecoder(n_genes)
    def forward(self, x, ei, ew, lib, tau):
        z                        = self.enc(x, ei, ew)
        z_hat, q, p, l_cb, l_cm = self.vq(z, tau)
        mu, theta                = self.dec(z_hat, lib)
        return z, q, p, mu, theta, l_cb, l_cm

def nb_loss(y, mu, theta, eps=1e-8):
    t1 = torch.lgamma(y+theta+eps) - torch.lgamma(theta+eps) - torch.lgamma(y+1.0)
    t2 = theta * torch.log(theta / (theta+mu+eps))
    t3 = y     * torch.log(mu    / (theta+mu+eps))
    return -(t1+t2+t3).mean()

def nb_loss_per_cell(y, mu, theta, eps=1e-8):
    t1 = torch.lgamma(y+theta+eps) - torch.lgamma(theta+eps) - torch.lgamma(y+1.0)
    t2 = theta * torch.log(theta / (theta+mu+eps))
    t3 = y     * torch.log(mu    / (theta+mu+eps))
    return -(t1+t2+t3).mean(dim=1)

def loss_spatial(p, ei, ew, eps=1e-8):
    chunk = 5000
    total, count = 0.0, ei.shape[1]
    for start in range(0, count, chunk):
        end  = min(start+chunk, count)
        pi_  = p[ei[0, start:end]]
        pj_  = p[ei[1, start:end]]
        m    = 0.5*(pi_+pj_)
        jsd  = (0.5*(pi_*(torch.log(pi_+eps)-torch.log(m+eps))).sum(-1)
              + 0.5*(pj_*(torch.log(pj_+eps)-torch.log(m+eps))).sum(-1))
        total += (ew[start:end]*jsd).sum()
    return total / count

def loss_usage(p, eps=1e-8):
    pb = p.mean(0)
    return (pb*torch.log(pb+eps)).sum()

def get_mini_batch(ei_S, ew_S, n):
    seed  = torch.randperm(n)[:B]
    hop1  = torch.unique(ei_S[1][torch.isin(ei_S[0], seed)])
    hop2  = torch.unique(ei_S[1][torch.isin(ei_S[0], hop1)])
    batch = torch.unique(torch.cat([seed, hop1, hop2]))
    mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
    ei_b  = ei_S[:, mask];  ew_b = ew_S[mask]
    lmap  = torch.zeros(n, dtype=torch.long)
    lmap[batch] = torch.arange(len(batch))
    return batch, lmap[ei_b].to(device), ew_b.to(device)

def fix_codebook(model, x, y, ei_S, ew_S, lib, n):
    model.eval()
    M = model.vq.M
    all_q, all_z, all_l = [], [], []
    with torch.no_grad():
        for s in range(0, n, 2048):
            batch = torch.arange(s, min(s+2048, n))
            mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
            ei_b  = ei_S[:, mask];  ew_b = ew_S[mask]
            lmap  = torch.zeros(n, dtype=torch.long)
            lmap[batch] = torch.arange(len(batch))
            z, q, _, mu, theta, _, _ = model(
                x[batch].to(device), lmap[ei_b].to(device),
                ew_b.to(device), lib[batch].to(device), tau=0.01)
            all_q.append(q.cpu()); all_z.append(z.cpu())
            all_l.append(nb_loss_per_cell(y[batch].to(device), mu, theta).cpu())
            torch.cuda.empty_cache()
    all_q = torch.cat(all_q); all_z = torch.cat(all_z); all_l = torch.cat(all_l)
    counts = torch.bincount(all_q, minlength=M)
    dead   = torch.where(counts < N_MIN)[0].tolist()
    big    = torch.where(counts > int(3*n/M))[0].tolist()
    for m in big:
        km = KMeans(2, n_init=1, random_state=0).fit(all_z[all_q==m].numpy())
        model.vq.codebook.data[m] = torch.tensor(km.cluster_centers_[0], dtype=torch.float32).to(device)
        if dead:
            d = dead.pop(0)
            model.vq.codebook.data[d] = torch.tensor(km.cluster_centers_[1], dtype=torch.float32).to(device)
            model.vq.N[d] = 1.0; model.vq.Sigma[d] = model.vq.codebook.data[d].clone()
    for m in dead:
        i      = int(all_l.argmax())
        z_star = all_z[i].to(device)
        noise  = torch.randn_like(z_star) * 1e-3
        model.vq.codebook.data[m] = z_star + noise
        model.vq.N[m] = 1.0; model.vq.Sigma[m] = model.vq.codebook.data[m].clone()
        all_l[i] = -float("inf")
    model.train()
    return int((counts < N_MIN).sum()), int((counts > int(3*n/M)).sum())

# ── load data ────────────────────────────────────────────────────────────────
data       = torch.load(f"{SAVE_DIR}/stage4_tensors.pt", weights_only=True)
anchor_idx = np.load(f"{SAVE_DIR}/stage5_anchors.npy")
x    = data["x_input"]; y = data["y_target"]
ei_S = data["edge_index_S"]; ew_S = data["edge_weight_S"]
lib  = y.sum(dim=1)
n, IN_DIM, G, M = x.shape[0], x.shape[1], y.shape[1], len(anchor_idx)
print(f"n={n}  in={IN_DIM}  genes={G}  M={M}  T_ANN={T_ANN}")

# ── check if checkpoint exists to resume training ────────────────────────────
CKPT_MODEL = f"{SAVE_DIR}/stage6_model.pt"
CKPT_OPT   = f"{SAVE_DIR}/stage6_optimizer.pt"
CKPT_META  = f"{SAVE_DIR}/stage6_checkpoint_meta.npy"
HIST_PATH  = f"{SAVE_DIR}/stage6_history.npy"

model = STAQ(IN_DIM, M, G).to(device)
params    = [p for nm, p in model.named_parameters() if "vq" not in nm]
optimizer = optim.Adam(params, lr=LR)

# REMOVED: scheduler = optim.lr_scheduler.ReduceLROnPlateau(...)

start_epoch = 1
history = {"epoch":[],"loss":[],"recon":[],"spatial":[],"dead":[],"mega":[],"tau":[],"lr":[]}

if os.path.exists(CKPT_MODEL) and os.path.exists(CKPT_META):
    print("\n>>> RESUMING from checkpoint <<<")
    model.load_state_dict(torch.load(CKPT_MODEL, weights_only=True))
    optimizer.load_state_dict(torch.load(CKPT_OPT, weights_only=True))
    meta = np.load(CKPT_META, allow_pickle=True).item()
    start_epoch = meta["epoch"] + 1
    if os.path.exists(HIST_PATH):
        history = np.load(HIST_PATH, allow_pickle=True).item()
    print(f"    resumed from epoch {meta['epoch']}  best_loss={meta['best_loss']:.4f}")
    best_loss = meta["best_loss"]
    model.train()
else:
    print("\n>>> FRESH training — initialising codebook from encoder outputs <<<")
    model.eval()
    anchor_t = torch.tensor(anchor_idx, dtype=torch.long)
    with torch.no_grad():
        hop   = torch.unique(ei_S[1][torch.isin(ei_S[0], anchor_t)])
        nodes = torch.unique(torch.cat([anchor_t, hop]))
        mask  = torch.isin(ei_S[0], nodes) & torch.isin(ei_S[1], nodes)
        ei_a  = ei_S[:, mask]; ew_a = ew_S[mask]
        lmap  = torch.zeros(n, dtype=torch.long)
        lmap[nodes] = torch.arange(len(nodes))
        z_a  = model.enc(x[nodes].to(device), lmap[ei_a].to(device), ew_a.to(device))
        E0   = z_a[lmap[anchor_t]].clone()
    model.vq.initialize(E0)
    del z_a, E0; torch.cuda.empty_cache()
    model.train()
    best_loss = float("inf")
    print("codebook initialised")

# ── training loop ────────────────────────────────────────────────────────────
steps_per_epoch = max(1, n // B)
loss_window     = []
prev_q          = None

print(f"\n{'ep':>5} {'loss':>8} {'recon':>8} {'sp':>7} {'tau':>6} {'dead':>5} {'mega':>5} {'a_sp':>6} {'lr':>8}")

for ep in range(start_epoch, EPOCHS+1):
    tau      = TAU0 * math.exp(-ep / T_ANN)
    a_sp_cur = A_SP * min(1.0, ep / SP_WARMUP)
    ep_losses, ep_lr, ep_lsp = [], [], []

    for _ in range(steps_per_epoch):
        batch, ei_bd, ew_bd = get_mini_batch(ei_S, ew_S, n)
        x_b = x[batch].to(device); y_b = y[batch].to(device); lb_b = lib[batch].to(device)
        optimizer.zero_grad()
        z, q, p, mu, theta, l_cb, l_cm = model(x_b, ei_bd, ew_bd, lb_b, tau)
        l_r  = nb_loss(y_b, mu, theta)
        l_sp = loss_spatial(p, ei_bd, ew_bd)
        l_ue = loss_usage(p)
        # loss = l_r + A_CB*l_cb + A_CM*l_cm + a_sp_cur*l_sp + A_UE*l_ue
        # remove SP_WARMUP entirely
        loss = l_r + A_CB*l_cb + A_CM*l_cm + A_SP*l_sp + A_UE*l_ue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        torch.cuda.empty_cache()
        ep_losses.append(loss.item()); ep_lr.append(l_r.item()); ep_lsp.append(l_sp.item())

    ep_loss = sum(ep_losses) / len(ep_losses)
    n_dead, n_mega = fix_codebook(model, x, y, ei_S, ew_S, lib, n)
    
    # REMOVED: scheduler.step(ep_loss)
    cur_lr  = LR # Use the static LR directly for logging
    
    avg_lr  = sum(ep_lr)  / len(ep_lr)
    avg_lsp = sum(ep_lsp) / len(ep_lsp)
    print(f"{ep:>5} {ep_loss:>8.4f} {avg_lr:>8.4f} {avg_lsp:>7.4f} "
        f"{tau:>6.4f} {n_dead:>5} {n_mega:>5} {a_sp_cur:>6.3f} {cur_lr:.2e}")

    # record history
    history["epoch"].append(ep); history["loss"].append(ep_loss)
    history["recon"].append(avg_lr); history["spatial"].append(avg_lsp)
    history["dead"].append(n_dead); history["mega"].append(n_mega)
    history["tau"].append(tau); history["lr"].append(cur_lr)

    # save checkpoint every 10 epochs
    if ep % 10 == 0 or ep_loss < best_loss:
        if ep_loss < best_loss:
            best_loss = ep_loss
        torch.save(model.state_dict(), CKPT_MODEL)
        torch.save(optimizer.state_dict(), CKPT_OPT)
        np.save(CKPT_META, {"epoch": ep, "best_loss": best_loss})
        np.save(HIST_PATH, history)

    # early stopping
    loss_window.append(ep_loss)
    if len(loss_window) > WINDOW: loss_window.pop(0)
    if len(loss_window) == WINDOW:
        loss_delta = abs(loss_window[0] - loss_window[-1])
        model.eval()
        cur_q = []
        with torch.no_grad():
            for s in range(0, n, 2048):
                batch = torch.arange(s, min(s+2048, n))
                mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
                ei_b  = ei_S[:, mask]; ew_b = ew_S[mask]
                lmap  = torch.zeros(n, dtype=torch.long)
                lmap[batch] = torch.arange(len(batch))
                _, q, _, _, _, _, _ = model(x[batch].to(device), lmap[ei_b].to(device),
                                            ew_b.to(device), lib[batch].to(device), tau=0.01)
                cur_q.append(q.cpu()); torch.cuda.empty_cache()
        cur_q = torch.cat(cur_q); model.train()
        if prev_q is not None:
            assign_delta = (cur_q != prev_q).float().mean().item()
            if loss_delta < LOSS_TOL and assign_delta < ASSIGN_TOL:
                print(f"\nearly stopping at epoch {ep} "
                      f"(loss_delta={loss_delta:.2e}, assign_delta={assign_delta:.3f})")
                break
        prev_q = cur_q

# ── final pass: save assignments, model, soft-p, codebook ───────────────────
print("\nrunning final inference pass ...")
model.eval()
all_q, all_z, all_p = [], [], []
with torch.no_grad():
    for s in range(0, n, 2048):
        batch = torch.arange(s, min(s+2048, n))
        mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
        ei_b  = ei_S[:, mask]; ew_b = ew_S[mask]
        lmap  = torch.zeros(n, dtype=torch.long)
        lmap[batch] = torch.arange(len(batch))
        z, q, p, _, _, _, _ = model(x[batch].to(device), lmap[ei_b].to(device),
                                     ew_b.to(device), lib[batch].to(device), tau=0.01)
        all_q.append(q.cpu()); all_z.append(z.cpu()); all_p.append(p.cpu())
        torch.cuda.empty_cache()

final_q = torch.cat(all_q); final_z = torch.cat(all_z); final_p = torch.cat(all_p)
torch.save(final_q, f"{SAVE_DIR}/stage6_assignments.pt")
torch.save(final_z, f"{SAVE_DIR}/stage6_embeddings.pt")
torch.save(final_p, f"{SAVE_DIR}/stage6_soft_assignments.pt")
torch.save(model.vq.codebook.detach().cpu(), f"{SAVE_DIR}/stage6_codebook.pt")
np.save(HIST_PATH, history)
print(f"metacells used: {final_q.unique().shape[0]}/{M}  best_loss={best_loss:.4f}")
print("[SUCCESS] stage6_assignments.pt, stage6_embeddings.pt,")
print("          stage6_soft_assignments.pt, stage6_codebook.pt, stage6_history.npy")

In [ ]:
# stage 7 - hard assignment + contiguity refinement
# PDF: reassign minority cells using argmax p_i(m') restricted to spatial neighbours
import os
import numpy as np
import torch
from collections import deque

SAVE_DIR = "/kaggle/working"
print("--- Stage 7: Hard Assignment + Contiguity Refinement ---")

data    = torch.load(os.path.join(SAVE_DIR, "stage4_tensors.pt"), weights_only=True)
final_q = torch.load(os.path.join(SAVE_DIR, "stage6_assignments.pt"), weights_only=True)
final_p = torch.load(os.path.join(SAVE_DIR, "stage6_soft_assignments.pt"), weights_only=True)
ei_S    = data["edge_index_S"]
ew_S    = data["edge_weight_S"]

n  = final_q.shape[0]
M  = int(final_q.max().item()) + 1
pi = final_q.numpy().copy()
print(f"n={n}  M={M}  unique before refinement={len(np.unique(pi))}")

neighbours = [[] for _ in range(n)]
nb_weights = [[] for _ in range(n)]
for s, d, w in zip(ei_S[0].numpy(), ei_S[1].numpy(), ew_S.numpy()):
    neighbours[s].append(int(d))
    nb_weights[s].append(float(w))

def connected_components(cell_list, neighbours):
    cell_set = set(cell_list); visited = set(); components = []
    for start in cell_list:
        if start in visited: continue
        comp = []; q = deque([start]); visited.add(start)
        while q:
            node = q.popleft(); comp.append(node)
            for nb in neighbours[node]:
                if nb in cell_set and nb not in visited:
                    visited.add(nb); q.append(nb)
        components.append(comp)
    return components

next_new_id = M; n_reassigned = 0; n_singletons = 0
for m in range(M):
    cell_list = np.where(pi == m)[0].tolist()
    if len(cell_list) <= 1: continue
    comps = connected_components(cell_list, neighbours)
    if len(comps) == 1: continue
    comps_sorted = sorted(comps, key=len, reverse=True)
    for comp in comps_sorted[1:]:
        for cell in comp:
            # PDF: argmax_{m'!=pi(i)} p_i(m') restricted to spatial neighbours
            adjacent = set()
            for nb in neighbours[cell]:
                if pi[nb] != m: adjacent.add(pi[nb])
            if adjacent:
                p_cell = final_p[cell]
                best_m = max(adjacent,
                             key=lambda m_: float(p_cell[m_]) if m_ < len(p_cell) else 0.0)
                pi[cell] = best_m
            else:
                pi[cell] = next_new_id; next_new_id += 1; n_singletons += 1
            n_reassigned += 1

print(f"cells reassigned: {n_reassigned}  singletons: {n_singletons}")
print(f"unique metacells after: {len(np.unique(pi))}  (M_final={next_new_id})")
pi_tensor = torch.tensor(pi, dtype=torch.long)
torch.save(pi_tensor, os.path.join(SAVE_DIR, "stage7_assignments.pt"))
np.save(os.path.join(SAVE_DIR, "stage7_assignments.npy"), pi)
print("[SUCCESS] saved stage7_assignments.pt")


In [ ]:
# stage 8 - aggregation
import os, numpy as np, torch, scanpy as sc
SAVE_DIR = "/kaggle/working"
print("--- Stage 8: Aggregation ---")

data  = torch.load(os.path.join(SAVE_DIR, "stage4_tensors.pt"), weights_only=True)
pi    = torch.load(os.path.join(SAVE_DIR, "stage7_assignments.pt"), weights_only=True).numpy()
adata = sc.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
y_raw = data["y_target"]
coords = adata.obsm["spatial"]
n, g  = y_raw.shape; ds = coords.shape[1]; M = int(pi.max()) + 1
print(f"n={n}  g={g}  M={M}")

Y = np.zeros((M, g), dtype=np.float32)
S_bar = np.zeros((M, ds), dtype=np.float32)
sizes = np.zeros(M, dtype=np.int64)
y_np = y_raw.numpy()
for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) == 0: continue
    sizes[m] = len(idx)
    Y[m]     = y_np[idx].sum(axis=0)
    S_bar[m] = coords[idx].mean(axis=0)

occ = (sizes > 0).sum()
print(f"occupied metacells: {occ}/{M}")
print(f"size min={sizes[sizes>0].min()}  median={int(np.median(sizes[sizes>0]))}  max={sizes.max()}")
print(f"counts match: {np.isclose(Y.sum(), y_np.sum())}")

np.save(os.path.join(SAVE_DIR, "stage8_Y.npy"), Y)
np.save(os.path.join(SAVE_DIR, "stage8_S_bar.npy"), S_bar)
np.save(os.path.join(SAVE_DIR, "stage8_sizes.npy"), sizes)
print("[SUCCESS] saved stage8_Y.npy, stage8_S_bar.npy, stage8_sizes.npy")


In [ ]:
# stage 9 - evaluation metrics (STAQ paper section 9)
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scanpy as sc
from collections import deque
from torch_geometric.nn import GATConv
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("--- Stage 9: Evaluation Metrics ---")

adata  = sc.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
data   = torch.load(os.path.join(SAVE_DIR, "stage4_tensors.pt"), weights_only=True)
pi     = torch.load(os.path.join(SAVE_DIR, "stage7_assignments.pt"), weights_only=True).numpy()
sizes  = np.load(os.path.join(SAVE_DIR, "stage8_sizes.npy"))
Z      = adata.obsm["X_pca"].astype("float32")
coords = adata.obsm["spatial"].astype("float32")
ei_S   = data["edge_index_S"]

n = len(pi); M = int(pi.max()) + 1
M_orig = len(np.load(os.path.join(SAVE_DIR, "stage5_anchors.npy")))
occ    = sizes > 0
print(f"n={n}  M={M}  M_orig={M_orig}")

# metric 1: transcriptomic compactness
print("\n[1] transcriptomic compactness kT ...")
kT = np.zeros(M)
for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) < 2: continue
    c = Z[idx].mean(axis=0)
    kT[m] = float(np.median(np.linalg.norm(Z[idx] - c, axis=1)))
print(f"  mean={kT[occ].mean():.4f}  median={np.median(kT[occ]):.4f}  max={kT[occ].max():.4f}")

# metric 2: spatial compactness
print("\n[2] spatial compactness kS ...")
kS = np.zeros(M)
for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) < 2: continue
    c = coords[idx].mean(axis=0)
    kS[m] = float(np.median(np.linalg.norm(coords[idx] - c, axis=1)))
print(f"  mean={kS[occ].mean():.4f}  median={np.median(kS[occ]):.4f}  max={kS[occ].max():.4f}")

# metric 3: niche entropy
print("\n[3] niche entropy ...")
niche_entropy = None; ct_col = None
for col in ["cell_type","celltype","CellType","cell_class","subclass","cluster","leiden","louvain"]:
    if col in adata.obs.columns: ct_col = col; break
if ct_col:
    labels = adata.obs[ct_col].values
    niche_entropy = np.zeros(M)
    for m in range(M):
        idx = np.where(pi == m)[0]
        if len(idx) == 0: continue
        counts = {}
        for c in labels[idx]: counts[c] = counts.get(c, 0) + 1
        total = len(idx)
        niche_entropy[m] = -sum((v/total)*math.log(v/total+1e-12) for v in counts.values())
    print(f"  mean={niche_entropy[occ].mean():.4f}  median={np.median(niche_entropy[occ]):.4f}")
else:
    print(f"  SKIPPED — no cell-type column. available: {list(adata.obs.columns)}")

# metric 4: purity — skipped (no ground-truth labels)
print("\n[4] purity — SKIPPED (no ground-truth labels)")

# metric 5: inner connectedness
print("\n[5] inner connectedness ...")
nb_list = [[] for _ in range(n)]
for s_, d_ in zip(ei_S[0].numpy(), ei_S[1].numpy()): nb_list[s_].append(int(d_))
def bfs(cell_list):
    cs, vis, comps = set(cell_list), set(), []
    for start in cell_list:
        if start in vis: continue
        comp, q = [], deque([start]); vis.add(start)
        while q:
            node = q.popleft(); comp.append(node)
            for nb in nb_list[node]:
                if nb in cs and nb not in vis: vis.add(nb); q.append(nb)
        comps.append(comp)
    return comps
n_single = 0; n_occ_cc = 0; n_comps = np.zeros(M, dtype=int)
for m in range(M):
    idx = np.where(pi == m)[0].tolist()
    if not idx: continue
    n_occ_cc += 1; comps = bfs(idx); n_comps[m] = len(comps)
    if len(comps) == 1: n_single += 1
ic = n_single / n_occ_cc if n_occ_cc else 0.0
print(f"  inner connectedness = {ic:.4f}  ({n_single}/{n_occ_cc})  [target > 0.95]")

# metrics 6 & 7: load saved embeddings (no model reload needed)
print("\n[6+7] loading saved embeddings ...")
all_z    = torch.load(os.path.join(SAVE_DIR, "stage6_embeddings.pt"),        weights_only=True).numpy()
all_p    = torch.load(os.path.join(SAVE_DIR, "stage6_soft_assignments.pt"),  weights_only=True)
codebook = torch.load(os.path.join(SAVE_DIR, "stage6_codebook.pt"),          weights_only=True).numpy()
all_q    = torch.load(os.path.join(SAVE_DIR, "stage6_assignments.pt"),       weights_only=True).numpy()

# metric 6: quantisation gap
print("\n[6] quantisation gap ...")
valid = pi < M_orig
mean_gap = float(np.sum((all_z[valid] - codebook[pi[valid]])**2, axis=1).mean())
rng = np.random.default_rng(42)
ii = rng.integers(0, M_orig, 5000); jj = rng.integers(0, M_orig, 5000)
jj[ii==jj] = (jj[ii==jj]+1) % M_orig
median_inter = float(np.median(np.linalg.norm(codebook[ii]-codebook[jj], axis=1)))
quant_gap = mean_gap / (median_inter + 1e-8)
print(f"  mean||z-e||^2={mean_gap:.4f}  inter-cb={median_inter:.4f}  gap={quant_gap:.4f} [<0.5]")

# metric 7: codebook usage balance
print("\n[7] codebook usage balance ...")
p_bar = all_p.mean(dim=0)
H_usage = -(p_bar * torch.log(p_bar+1e-12)).sum().item()
H_max   = math.log(M_orig)
balance = H_usage / H_max
print(f"  H(p_bar)={H_usage:.4f}  log(M)={H_max:.4f}  balance={balance:.4f} [>0.8]")

print("\n"+"="*50)
print("  STAQ EVALUATION METRICS")
print("="*50)
print(f"  [1] kT  transcriptomic compactness : {kT[occ].mean():.4f}")
print(f"  [2] kS  spatial compactness        : {kS[occ].mean():.4f}")
print(f"  [3]     niche entropy              : {niche_entropy[occ].mean():.4f}" if niche_entropy is not None else "  [3]     niche entropy              : N/A")
print(f"  [5]     inner connectedness        : {ic:.4f}  [>0.95]")
print(f"  [6]     quantisation gap           : {quant_gap:.4f}  [<0.50]")
print(f"  [7]     codebook usage balance     : {balance:.4f}  [>0.80]")
print("="*50)

out = dict(kT=kT, kS=kS, n_components=n_comps,
           inner_connectedness=np.array([ic]),
           quant_gap=np.array([quant_gap]),
           usage_balance=np.array([balance]),
           metacell_sizes=sizes)
if niche_entropy is not None: out["niche_entropy"] = niche_entropy
np.savez(os.path.join(SAVE_DIR, "stage9_metrics.npz"), **out)
print("[SUCCESS] saved stage9_metrics.npz")


## Visualisations

In [ ]:
# visualisations — setup
import os, math
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scanpy as sc
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    "figure.dpi": 150, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
})

SAVE_DIR = "/kaggle/working"
adata    = sc.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
coords   = adata.obsm["spatial"].astype("float32")
Z        = adata.obsm["X_pca"].astype("float32")
pi       = torch.load(os.path.join(SAVE_DIR, "stage7_assignments.pt"), weights_only=True).numpy()
sizes    = np.load(os.path.join(SAVE_DIR, "stage8_sizes.npy"))
S_bar    = np.load(os.path.join(SAVE_DIR, "stage8_S_bar.npy"))
Y        = np.load(os.path.join(SAVE_DIR, "stage8_Y.npy"))
hist     = np.load(os.path.join(SAVE_DIR, "stage6_history.npy"), allow_pickle=True).item()
metrics  = np.load(os.path.join(SAVE_DIR, "stage9_metrics.npz"), allow_pickle=True)
n        = len(pi); M = int(pi.max()) + 1; occ = sizes > 0
epochs   = hist["epoch"]
print(f"n={n}  M={M}  occupied={occ.sum()}  epochs_trained={len(epochs)}")


In [ ]:
# plot 1: training loss curves
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("STAQ Training Curves", fontsize=14, fontweight="bold")

ax = axes[0,0]
ax.plot(epochs, hist["loss"], color="#2563eb", lw=1.5)
ax.set_title("Total Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_yscale("log")

ax = axes[0,1]
ax.plot(epochs, hist["recon"], color="#16a34a", lw=1.5)
ax.set_title("Reconstruction Loss $\mathcal{L}_{recon}$")
ax.set_xlabel("Epoch"); ax.set_ylabel("NB NLL"); ax.set_yscale("log")

ax = axes[0,2]
ax.plot(epochs, hist["spatial"], color="#dc2626", lw=1.5)
ax.set_title("Spatial Smoothness Loss $\mathcal{L}_{spatial}$")
ax.set_xlabel("Epoch"); ax.set_ylabel("JSD")

ax = axes[1,0]
ax.plot(epochs, hist["dead"],  color="#7c3aed", lw=1.5, label="Dead codes")
ax.plot(epochs, hist["mega"],  color="#ea580c", lw=1.5, ls="--", label="Mega codes")
ax.set_title("Codebook Health"); ax.set_xlabel("Epoch"); ax.set_ylabel("# Codes"); ax.legend()

ax = axes[1,1]
ax.plot(epochs, hist["tau"], color="#0891b2", lw=1.5)
ax.set_title("Temperature $\tau$ Annealing"); ax.set_xlabel("Epoch"); ax.set_ylabel("$\tau$")

ax = axes[1,2]
ax.plot(epochs, hist["lr"], color="#475569", lw=1.5)
ax.set_title("Learning Rate"); ax.set_xlabel("Epoch"); ax.set_ylabel("LR"); ax.set_yscale("log")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot1_training_curves.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot1_training_curves.png")


In [ ]:
# plot 2: tissue map
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("STAQ Spatial Metacell Assignment", fontsize=13, fontweight="bold")

ax = axes[0]
ax.scatter(coords[:,0], coords[:,1], c=pi, cmap="tab20", s=1.5, alpha=0.6, linewidths=0)
ax.set_title(f"Cells coloured by metacell (n={n:,}, M={M})"); ax.axis("off"); ax.set_aspect("equal")

ax = axes[1]
ax.scatter(coords[:,0], coords[:,1], c="#cbd5e1", s=1, alpha=0.2, linewidths=0)
occ_idx = np.where(occ)[0]
ax.scatter(S_bar[occ_idx,0], S_bar[occ_idx,1], c=occ_idx, cmap="tab20",
           s=40, marker="*", edgecolors="black", linewidths=0.3, alpha=0.9)
ax.set_title(f"Metacell centroids ({occ.sum()} occupied)"); ax.axis("off"); ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot2_tissue_map.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot2_tissue_map.png")


In [ ]:
# plot 3: metacell size distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Metacell Size Distribution", fontsize=13, fontweight="bold")
occ_sizes = sizes[occ]; target = n / occ.sum()

ax = axes[0]
ax.hist(occ_sizes, bins=40, color="#3b82f6", edgecolor="white", lw=0.4)
ax.axvline(target, color="#dc2626", ls="--", lw=1.5, label=f"Target n/M={target:.0f}")
ax.axvline(np.median(occ_sizes), color="#16a34a", ls="--", lw=1.5,
           label=f"Median={np.median(occ_sizes):.0f}")
ax.set_xlabel("Cells per metacell"); ax.set_ylabel("Count"); ax.set_title("Size histogram"); ax.legend()

ax = axes[1]
sorted_s = np.sort(occ_sizes); cdf = np.arange(1, len(sorted_s)+1)/len(sorted_s)
ax.plot(sorted_s, cdf, color="#3b82f6", lw=1.5)
ax.axvline(target, color="#dc2626", ls="--", lw=1.5, label=f"Target={target:.0f}")
ax.set_xlabel("Cells per metacell"); ax.set_ylabel("Cumulative fraction"); ax.set_title("CDF"); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot3_sizes.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot3_sizes.png")


In [ ]:
# plot 4: QC metrics
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("STAQ QC Metrics (Paper Section 9)", fontsize=13, fontweight="bold")
kT_v = metrics["kT"]; kS_v = metrics["kS"]
ic   = float(metrics["inner_connectedness"])
qg   = float(metrics["quant_gap"])
ub   = float(metrics["usage_balance"])

ax = axes[0]
ax.boxplot([kT_v[occ], kS_v[occ]], labels=["$\kappa_T$ (PCA)","$\kappa_S$ (Physical)"],
           patch_artist=True, boxprops=dict(facecolor="#bfdbfe"),
           medianprops=dict(color="#1e3a8a", lw=2))
ax.set_title("Compactness (lower=better)"); ax.set_ylabel("Median intra-metacell distance")

ax = axes[1]
colors = ["#ef4444" if qg>=0.5 else "#22c55e", "#ef4444" if ub<=0.8 else "#22c55e"]
bars = ax.bar(["Quant Gap\n[<0.5]","Usage Balance\n[>0.8]"], [qg,ub],
              color=colors, width=0.5, edgecolor="white")
ax.axhline(0.5, color="#dc2626", ls="--", lw=1, alpha=0.6)
ax.axhline(0.8, color="#16a34a", ls="--", lw=1, alpha=0.6)
ax.set_ylim(0, max(1.0, qg*1.1)); ax.set_title("VQ-Diagnostic Metrics")
for bar, val in zip(bars,[qg,ub]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax = axes[2]
ax.pie([ic, 1-ic], labels=[f"Contiguous\n{ic:.1%}", f"Fragmented\n{1-ic:.1%}"],
       colors=["#22c55e","#f87171"], startangle=90,
       wedgeprops=dict(edgecolor="white", linewidth=1.5))
ax.set_title(f"Inner Connectedness [>0.95]")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot4_qc.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot4_qc.png")


In [ ]:
# plot 5: PCA embedding coloured by metacell
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Latent Space Structure", fontsize=13, fontweight="bold")

ax = axes[0]
ax.scatter(Z[:,0], Z[:,1], c=pi, cmap="tab20", s=1, alpha=0.4, linewidths=0)
ax.set_title("Cells in PCA space\ncoloured by metacell"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_aspect("equal")

mc_pca = np.zeros((M, Z.shape[1]))
for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) > 0: mc_pca[m] = Z[idx].mean(axis=0)

ax = axes[1]
ax.scatter(mc_pca[occ,0], mc_pca[occ,1], c=np.where(occ)[0], cmap="tab20",
           s=sizes[occ]*0.5, alpha=0.7, edgecolors="white", linewidths=0.3)
ax.set_title("Metacell centroids in PCA\n(size ∝ metacell size)"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot5_pca.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot5_pca.png")


In [ ]:
# plot 6: codebook health over training + final assignment distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Codebook Health", fontsize=13, fontweight="bold")
M_orig = len(np.load(os.path.join(SAVE_DIR, "stage5_anchors.npy")))

ax = axes[0]
dead_pct = [d/M_orig*100 for d in hist["dead"]]
mega_pct = [m/M_orig*100 for m in hist["mega"]]
ax.fill_between(epochs, dead_pct, alpha=0.3, color="#7c3aed")
ax.plot(epochs, dead_pct, color="#7c3aed", lw=1.5, label="Dead codes %")
ax.fill_between(epochs, mega_pct, alpha=0.3, color="#ea580c")
ax.plot(epochs, mega_pct, color="#ea580c", lw=1.5, ls="--", label="Mega codes %")
ax.set_xlabel("Epoch"); ax.set_ylabel("% of codebook")
ax.set_title("Dead & Mega codes (both → 0 = healthy)"); ax.legend(); ax.set_ylim(bottom=0)

ax = axes[1]
ac = np.bincount(pi, minlength=M); sc2 = np.sort(ac[occ])[::-1]
ax.bar(range(len(sc2)), sc2, color="#3b82f6", edgecolor="none", alpha=0.7)
ax.axhline(n/occ.sum(), color="#dc2626", ls="--", lw=1.5, label=f"Ideal={n/occ.sum():.0f}")
ax.set_xlabel("Metacell rank"); ax.set_ylabel("Cells assigned")
ax.set_title("Assignment distribution (flat=uniform)"); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot6_codebook.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot6_codebook.png")


In [ ]:
# plot 7: pseudobulk heatmap
fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle("Pseudobulk Expression — Top 30 Metacells by Size", fontsize=13, fontweight="bold")
top30 = np.argsort(sizes)[::-1][:30]
Y_top = Y[top30]; Y_norm = Y_top / (Y_top.sum(axis=1, keepdims=True)+1e-8)
top_genes = np.argsort(Y_norm.var(axis=0))[::-1][:40]
im = ax.imshow(Y_norm[:,top_genes].T, aspect="auto", cmap="viridis", interpolation="nearest")
plt.colorbar(im, ax=ax, label="Normalised expression", shrink=0.8)
ax.set_xlabel("Metacell (ranked by size)"); ax.set_ylabel("Gene (top 40 by variance)")
ax.set_xticks(range(30)); ax.set_xticklabels([f"MC{i}" for i in top30], rotation=45, ha="right", fontsize=7)
ax.set_title("Each column=metacell pseudobulk  |  Each row=gene")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plot7_heatmap.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot7_heatmap.png")


In [ ]:
# plot 8: summary dashboard
fig = plt.figure(figsize=(14, 4))
fig.suptitle("STAQ Results Summary", fontsize=14, fontweight="bold", y=1.02)
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.5)
ic  = float(metrics["inner_connectedness"])
qg  = float(metrics["quant_gap"])
ub  = float(metrics["usage_balance"])
kT_mean = float(metrics["kT"][occ].mean())

def metric_box(ax, value, label, target_str, good):
    color = "#22c55e" if good else "#ef4444"
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    ax.add_patch(plt.Rectangle((0.05,0.05),0.9,0.9, facecolor=color+"22",
                                edgecolor=color, linewidth=2, transform=ax.transAxes))
    ax.text(0.5,0.68, f"{value:.3f}", ha="center", fontsize=20, fontweight="bold",
            color=color, transform=ax.transAxes)
    ax.text(0.5,0.44, label, ha="center", fontsize=9, color="#1e293b", transform=ax.transAxes)
    ax.text(0.5,0.22, target_str, ha="center", fontsize=8, color="#64748b", transform=ax.transAxes)
    ax.text(0.5,0.08, "✓" if good else "✗", ha="center", fontsize=12, color=color, transform=ax.transAxes)

metric_box(fig.add_subplot(gs[0]), ic,     "Inner\nConnectedness",    "target > 0.95", ic>0.95)
metric_box(fig.add_subplot(gs[1]), qg,     "Quantisation\nGap",       "target < 0.50", qg<0.50)
metric_box(fig.add_subplot(gs[2]), ub,     "Codebook Usage\nBalance", "target > 0.80", ub>0.80)
metric_box(fig.add_subplot(gs[3]), kT_mean,"Mean kT\n(Compactness)",  "lower = better", True)

plt.savefig(os.path.join(SAVE_DIR, "plot8_dashboard.png"), dpi=200, bbox_inches="tight")
plt.show(); print("saved plot8_dashboard.png")
print()
print("="*55); print("  ALL PLOTS COMPLETE"); print("="*55)
print(f"  Epochs trained       : {len(epochs)}")
print(f"  Inner connectedness  : {ic:.4f}  {'✓' if ic>0.95 else '✗'}")
print(f"  Quantisation gap     : {qg:.4f}  {'✓' if qg<0.5 else '✗'}")
print(f"  Usage balance        : {ub:.4f}  {'✓' if ub>0.8 else '✗'}")
print(f"  kT mean              : {kT_mean:.4f}")
print("="*55)
